In [1]:
# imports

import os
import openai
import logging
openai.api_key = os.environ["OPENAI_API_KEY"]

In [2]:
# general definitions and helper functions

SPLITTER = '.'

# questions per fact
n_questions = 'a'

# generations per question
n_regenerate = 4


def oai_predict(prompt):
    if isinstance(prompt, str):
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ]
    else:
        messages = prompt
    
    output = openai.ChatCompletion.create(
        model='gpt-3.5-turbo',
        messages=messages,
        max_tokens=200,
    )
    response = output['choices'][0]['message']['content']
    return response


def log_w_indent(text, indent):
    logging.info((indent * 2) * '>>' + ' ' + text)

def predict_w_log(prompt, indent):
    log_w_indent(f'Input: {prompt}', indent)
    response = oai_predict(prompt)
    log_w_indent(f'Output: {response}', indent)
    return response

def setup_logger():
    """Setup logger to always print time and level."""
    logging.basicConfig(
        format='%(asctime)s %(levelname)-8s %(message)s',
        level=logging.INFO,
        datefmt='%Y-%m-%d %H:%M:%S')
    logging.getLogger().setLevel(logging.INFO)  # logging.DEBUG
setup_logger()

def divider(symbol = '*'):
    logging.info(80 * symbol)

base_original_input = '{entity_type} is {entity}?'
base_initial_prompt = "{original_input} Please answer in concise sentences that contain single facts. Provide as much information as possible."
base_gen_questions_prompt = 'The following sentence contains facts about {entity}. Generate {n_questions} questions about {entity} that can be answered from the following sentence: {fact}'

base_answer_question_prompt = 'Please respond as concisely as possible to the following question about {entity}: {question}'

base_equivalence_prompt = "The following sentence contains a fact about {entity}: {fact}\nAre the following sentences the same as this fact?"
# one extra for the fact
for i in range(1, n_regenerate + 1):
    base_equivalence_prompt += f'\n{i}. ' + '{}'
base_equivalence_prompt += "\nRespond with yes or no."

In [3]:
results = dict()
results['prompts'] = dict(
    base_original_input=base_original_input,
    base_initial_prompt = base_initial_prompt,
    base_gen_questions_prompt = base_gen_questions_prompt,
    base_answer_question_prompt = base_answer_question_prompt,
    base_equivalence_prompt = base_equivalence_prompt,
)
logging.info(f'Using prompts {results["prompts"]}')

2023-09-26 13:16:28 INFO     Using prompts {'base_initial_prompt': '{entity_type} is {entity}? Please answer in concise sentences that contain single facts. Provide as much information as possible.', 'base_gen_questions_prompt': 'The following sentence contains facts about {entity}. Generate {n_questions} questions about {entity} that can be answered from the following sentence: {fact}', 'base_answer_question_prompt': 'Please respond as concisely as possible to the following question about {entity}: {question}', 'base_equivalence_promp': 'The following sentence contains a fact about {entity}: {fact}\nAre the following sentences the same as this fact?\n1. {}\n2. {}\n3. {}\n4. {}\nRespond with yes or no.'}


In [4]:
# replace this with loop over entities
entity = 'Yarin Gal'
entity_type = 'Who'
results[entity] = dict()
e_results = results[entity]
e_results['entity_type'] = entity_type

In [7]:
divider()
log_w_indent(f'Starting with entity {entity}, type {entity_type}', 0)

2023-09-26 14:10:49 INFO     ********************************************************************************
2023-09-26 14:10:49 INFO      Starting with entity Yarin Gal, type Who


In [9]:
initial_question = base_initial_question.format(entity_type=entity_type, entity=entity)
base_initial_prompt.format(initial_question=initial_question)
e_results['initial'] = predict_w_log(base_initial_prompt, indent)

NameError: name 'base_initial_question' is not defined

In [ ]:
# split response into facts
facts = [(r + SPLITTER) for r in e_results['initial'].split(SPLITTER) if r]
facts = [f.replace('\n', '').strip() for f in facts]
results[entity]['facts'] = facts
log_w_indent(f'Extracted facts: {facts}', 0)

In [ ]:
# replace this with loop over facts
fidx = 4
fact = facts[fidx]

log_w_indent(f'Currently dealing with fact {fidx}: {fact}', 1)

In [ ]:
e_results['questions'] = {f'fact-{fidx}':  {}}
gen_questions = predict_w_log(base_gen_questions_prompt.format(entity=entity, n_questions=n_questions, fact=fact), indent)

In [ ]:
questions = [q[3:] for q in gen_questions.split('\n')]
questions = [i for i in questions if i]
log_w_indent(f'Extracted questions: {questions}', 1)

In [ ]:
# replace with a loop over questions
qidx = 1

question = questions[qidx]
e_results['questions'][f'fact-{fidx}'][f'question-{qidx}'] = {'question': question, 'answers': []}

In [ ]:
answers = e_results['questions'][f'fact-{fidx}'][f'question-{qidx}']['answers']
log_w_indent('Regenerate answers for fact:', 1)
for re_gen in range(n_regenerate):
    answer = predict_w_log(base_answer_question_prompt.format(entity=entity, question=question), 2)
    answers.append(answer)

In [ ]:
equiv_prompt = base_equivalence_prompt.format(entity=entity, fact=fact, *answers)
print(equiv_prompt)

In [ ]:
equiv_prompt = base_equivalence_prompt.format(entity=entity, fact=fact, *answers)
equiv_response = predict_w_log(equiv_prompt, 1)

In [ ]:
e_results['final'] = []

In [ ]:
if 'yes' in equiv_response.lower():
    e_results.final.append(fact)
elif 'no' in equiv_response.lower():
    most_likely_answer = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": equiv_prompt},
            {"role": "system", "content": equiv_response},
            {"role": "user", "content": f"Then, based on the above, what is the most likely fact about {entity}?"},
        ]
    logging.info(


    
else:
    # how to handle this?
    raise

In [ ]:
equiv_response

In [ ]:
question_dict['answers'] = []
for regen in range(n_regenerate):
    answer_question_response = oai_predict(answer_question_prompt)
    question_dict['answers'].append(answer_question_response)
    print(answer_question_response)

In [ ]:


equivalence_prompt = base_equivalence_prompt
for i, answer in enumerate(question_dict['answers']):
    equivalence_prompt += f'\n{i}. {answer}'

print(equivalence_prompt)